# **Important Rules and Restrictions**

Use of the following libraries is strictly prohibited:

- NLTK  
- spaCy  
- HuggingFace Tokenizers  
- SentencePiece  
- Urduhack  
- Stanza  
- Polyglot  
- Gensim Language Models  
- sklearn CountVectorizer / TfidfVectorizer  

Violation will result in zero marks.

### **Allowed Libraries**
Students may use:

- Python Standard Functions 
- Regex  
- NumPy  
- Pandas  
- BeautifulSoup / Requests / Scrapy / Selenium (For use of Scraping only)  
- Matplotlib  

## **Part 1: BBC Urdu Dataset Collection and Preprocessing**
In this part, BBC Urdu news articles are collected and prepared into a clean dataset using normalization and custom preprocessing tools. The output of this part will be reused directly in Part 2.

News articles must be scraped from:
https://www.bbc.com/urdu

Students must scrape:
- Minimum: 200 articles
- Maximum: 300 articles

Each article must be complete and properly structured.

Article metadata must be stored in a JSON file with the following constraints:
- Each article must be numbered
- Article numbers must be unique
- Article numbers must match TXT files
- Article body must not be included

### ***Format Example***
```json
{
  "1": {
    "title": "پاکستان میں مہنگائی کی شرح میں اضافہ",
    "publish_date": "2024-01-15"
  },
  "2": {
    "title": "کراچی میں بارش کے بعد صورتحال",
    "publish_date": "2024-02-02"
  }
}

<b>Data Collection: Web Scraping for BBC Urdu Articles</b>

In [19]:
# scraping the whole website for articles 
import requests
from bs4 import BeautifulSoup
import time

BASE_URL = "https://www.bbc.com"     # using the link as a base for scraping       
MAX_ARTICLES = 275       # total articles to scrap 
articles_data = []       # stores data from the articles scraped
article_links = []       # storing URL links of articles for metajson file 

# BBC Urdu topic page for collecting different variety of articles 
section_urls = [
    "https://www.bbc.com/urdu/topics/c40379e2ymxt",   # Science link
    "https://www.bbc.com/urdu/topics/ckdxnx900n5t",   # Entertainment link
    "https://www.bbc.com/urdu/topics/c340q0p2585t",   # Sports link
    "https://www.bbc.com/urdu/topics/cw57v2pmll9t",   # World link
    "https://www.bbc.com/urdu/topics/cl8l9mveql2t",   # Near link
    "https://www.bbc.com/urdu/topics/cjgn7n9zzq7t"    # Pakistan link
]


# Collect article URLs from multiple BBC Urdu section pages
# Ensure links are valid, unique, and reachable

headers = { "User-Agent": "Mozilla/5.0"}  # To avoid request blocking and ensure stable scraping

for section in section_urls:
    print(f"Collecting links from:{section}")

    try:
        response = requests.get(section, headers=headers, timeout=10)
        response.raise_for_status()
    except Exception as e:
        print(f"Failed to load section: {e}")
        continue

    soup = BeautifulSoup(response.text, "html.parser")

    for a in soup.find_all("a", href=True):
        href = a["href"]

        if "/urdu/articles/" in href:
            if href.startswith("http"):
                full_url = href
            else:
                full_url = BASE_URL + href

            if full_url not in article_links:
                article_links.append(full_url)

print("Total unique article links collected:", len(article_links))


# scraping articles 

article_number = 1

for url in article_links:
    if article_number > MAX_ARTICLES:    # base for the for loop
        break

    print(f"Scraping article {article_number}")

    try:                                                          # connecting with the URL website using requests
        page = requests.get(url, headers=headers, timeout=10)
        page.raise_for_status()
    except Exception as e:
        print(f"Skipping article due to error: {e}")
        continue

    soup = BeautifulSoup(page.text, "html.parser")               # using soap to parse the html of the article

    # extracting title from the article
    title_tag = soup.find("h1")
    title = title_tag.text.strip() if title_tag else "N/A"

    # gathering date of the article 
    date_tag = soup.find("time")
    date = date_tag.text.strip() if date_tag else "N/A"

    # data for raw.txt 
    paragraphs = soup.find_all("p")
    body_text = ""

    for p in paragraphs:
        text = p.text.strip()
        if text:
            body_text+=text+"\n"

    if len(body_text.strip())<200:
        continue
    # appends the gathered data into the temporary data storage section
    articles_data.append({ "number": article_number,"title": title,"date": date,"url": url,"body": body_text})    

    article_number+= 1
    time.sleep(2) 


Total unique article links collected: 113
Scraping article 1
Scraping article 2
Scraping article 3
Scraping article 4
Scraping article 5
Scraping article 6
Scraping article 7
Scraping article 8
Scraping article 9
Scraping article 10
Scraping article 11
Scraping article 12
Scraping article 13
Scraping article 14
Scraping article 15
Scraping article 16
Scraping article 17
Scraping article 18
Scraping article 19
Scraping article 20
Scraping article 21
Scraping article 22
Scraping article 23
Scraping article 24
Scraping article 25
Scraping article 26
Scraping article 27
Scraping article 28
Scraping article 29
Scraping article 30
Scraping article 31
Scraping article 32
Scraping article 33
Scraping article 34
Scraping article 35
Scraping article 36
Scraping article 37
Scraping article 38
Scraping article 39
Scraping article 40
Scraping article 41
Scraping article 42
Scraping article 43
Scraping article 44
Scraping article 45
Scraping article 46
Scraping article 47
Scraping article 48
Scrapin

<b>Continuation of above web scraping content to load rest of articles </b>

In [29]:
import json
import requests
from bs4 import BeautifulSoup
import time

BASE_URL= "https://www.bbc.com"
TARGET_ARTICLES =285

# Load existing metadata to cintinue scraping for remaining articles
with open("Metadata.json", "r", encoding="utf-8") as f:
    metadata=json.load(f)

existing_urls = set(meta["url"] for meta in metadata.values())  ## Track already-scraped URLs to avoid duplication
article_number = max(map(int, metadata.keys())) + 1     # Continue article numbering

print("Starting extension from article:", article_number)

headers = {"User-Agent": "Mozilla/5.0"}   # Request headers to avoid blocking

topic_urls = [
    "https://www.bbc.com/urdu/topics/c40379e2ymxt",
    "https://www.bbc.com/urdu/topics/ckdxnx900n5t",
    "https://www.bbc.com/urdu/topics/c340q0p2585t",
    "https://www.bbc.com/urdu/topics/cw57v2pmll9t",
    "https://www.bbc.com/urdu/topics/cl8l9mveql2t",
    "https://www.bbc.com/urdu/topics/cjgn7n9zzq7t"
]

# Continuing scraping for new articles 
for topic in topic_urls:
    page = 1
    while article_number <= TARGET_ARTICLES:
        page_url = f"{topic}?page={page}"
        print(f"Scanning {page_url}")

        try:
            r = requests.get(page_url, headers=headers, timeout=10)
            r.raise_for_status()
        except:
            break

        soup = BeautifulSoup(r.text, "html.parser")
        found_any = False

        for a in soup.find_all("a", href=True):   # Scan all links on page
            if article_number>TARGET_ARTICLES:
                break

            href = a["href"]
            if "/urdu/articles/" not in href: # Only consider BBC Urdu article links
                continue

            url = href if href.startswith("http") else BASE_URL + href
            # Skip already scraped articles
            if url in existing_urls:
                continue

            try:
                art = requests.get(url, headers=headers, timeout=10)
                art.raise_for_status()
            except:
                continue
            # Extract article body
            art_soup = BeautifulSoup(art.text, "html.parser")
            paragraphs = art_soup.find_all("p")
            body = "\n".join(p.text.strip() for p in paragraphs if p.text.strip())

            if len(body)<200:
                continue
            # Save raw article text
            # append to raw.txt
            with open("raw.txt", "a", encoding="utf-8") as f:
                f.write(f"[{article_number}]\n{body}\n\n")

            metadata[str(article_number)] = {
                "title": art_soup.find("h1").text.strip() if art_soup.find("h1") else "N/A",
                "date": art_soup.find("time").text.strip() if art_soup.find("time") else "N/A",
                "url": url
            }

            existing_urls.add(url)
            print(f"Added article {article_number}")
            article_number += 1
            found_any = True
            time.sleep(2)

        if not found_any:
            break

        page += 1

# Saving metadata 
with open("Metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Finished. Total articles:",article_number-1)


Starting extension from article: 114
Scanning https://www.bbc.com/urdu/topics/c40379e2ymxt?page=1
Scanning https://www.bbc.com/urdu/topics/ckdxnx900n5t?page=1
Scanning https://www.bbc.com/urdu/topics/c340q0p2585t?page=1
Added article 114
Scanning https://www.bbc.com/urdu/topics/c340q0p2585t?page=2
Added article 115
Added article 116
Added article 117
Added article 118
Added article 119
Added article 120
Added article 121
Added article 122
Added article 123
Added article 124
Added article 125
Added article 126
Added article 127
Added article 128
Added article 129
Added article 130
Added article 131
Added article 132
Added article 133
Added article 134
Added article 135
Added article 136
Added article 137
Added article 138
Scanning https://www.bbc.com/urdu/topics/c340q0p2585t?page=3
Added article 139
Added article 140
Added article 141
Added article 142
Added article 143
Added article 144
Added article 145
Added article 146
Added article 147
Added article 148
Added article 149
Added arti

In [27]:
# Your code here, Scrape BBC Urdu articles and store metadata in JSON format
import json
# Dictionary to store metadata for all articles
metadata = {}
for article in articles_data:
    num = article["number"]
    # Store only metadata 
    metadata[str(num)] = {"title": article["title"],"date": article["date"],"url": article["url"]}
# Writing metadata to JSON file
with open("Metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Metadata.json created successfully")

Metadata.json created successfully


### ***raw.txt File***
This file must contain:
- Raw scraped article content
- No cleaning or normalization
- One article per block
- Each article must start with its article number

### ***Example***
```json
[1]
یہ خبر بی بی سی اردو سے حاصل کی گئی ہے...

[2]
کراچی میں بارش کے بعد صورتحال خراب ہو گئی...

In [28]:
# Your code here, Save raw scraped articles into raw.txt

# Writing raw data into txt file 
with open("raw.txt", "w", encoding="utf-8") as f:
    for article in articles_data:
        f.write(f"[{article['number']}]\n")
        f.write(article["body"] + "\n\n")

print("raw.txt created successfully")

raw.txt created successfully


## ***cleaned.txt File***

This file must contain:
- Fully preprocessed data
- Normalized Urdu text
- Noise removed
- Sentence segmented
- Ready for language model training
- Article numbering matching raw.txt and JSON

**Refer to the given Assignment PDF Document for the data cleaning and normalizing techniques**

In [2]:
# Your code here, Save clean scraped articles into cleaned.txt

# Data Cleaning and Normalization for Urdu Text
import re

# Read raw articles
with open("raw.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

#Defining cleaning functions
# Remove Urdu diacritics
urdu_diacritics = re.compile(r"[\u064B-\u065F\u0670\u06D6-\u06ED]")
def remove_diacritics(text):
    return re.sub(urdu_diacritics, "", text)

#Remove URLs, emojis, symbols
def remove_noise(text):
    # Removing urls 
    text = re.sub(r"http\S+|www\S+", "", text)

    # Removing emojis and symbols
    text=re.sub(r"[^\u0600-\u06FF\u0660-\u0669۔،؟!\n ]", " ", text)

    return text
# Remove English words 
def keep_only_urdu(text):
    return re.sub(r"[A-Za-z]+", "", text)

# Replace numbers with <NUM>
def normalize_numbers(text):
    return re.sub(r"\d+", "<NUM>", text)

# Sentence segmentation using Urdu punctuation
def sentence_segment(text):
    sentences = re.split(r"(؟|!|۔)", text)
    output=[]

    for i in range(0, len(sentences)-1, 2):
        sentence = sentences[i].strip()
        punct = sentences[i+1]
        if sentence:
            output.append(sentence + punct)

    return "\n".join(output)

# Normalize whitespace
def normalize_whitespace(text):
    text =re.sub(r"\s+", " ", text)
    text= re.sub(r"\n\s+", "\n", text)
    return text.strip()

def tokenize(text):
    return text.split()

def stem(tokens):
    suffixes= ["وں", "یں", "ات", "گی", "گا", "نے", "کا", "کی", "کے"]
    stemmed =[]

    for word in tokens:
        for suf in suffixes:
            if word.endswith(suf):
                word = word[:-len(suf)]
        stemmed.append(word)

    return stemmed

def lemmatize(tokens):
    lemma_map = {
        "لڑکیاں": "لڑکی",
        "بچے": "بچہ",
        "عورتیں": "عورت"
    }
    return [lemma_map.get(w, w) for w in tokens]

In [3]:
# Process articles one by one using the functions defined above 

cleaned_articles=[]

# splitting articles using article numbers 
articles = re.split(r"\[\d+\]", raw_text)[1:]
article_numbers = re.findall(r"\[(\d+)\]", raw_text)

# performing normalization and cleaning data using functions 
for num, article in zip(article_numbers, articles):
    text = remove_diacritics(article)
    text = remove_noise(text)
    text = keep_only_urdu(text)
    text = normalize_numbers(text)
    text = normalize_whitespace(text)
    text = sentence_segment(text)
    cleaned_articles.append(f"[{num}]\n{text}\n")

# writing the final output in cleaned text
with open("cleaned.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(cleaned_articles))
print("cleaned.txt generated successfully with", len(cleaned_articles), "articles")

cleaned.txt generated successfully with 285 articles


## ***Custom Urdu Tokenizer***

The tokenizer must handle:
- Word boundaries
- Punctuation
- Postpositions
- Numbers and special tokens

All numbers must be replaced with `<NUM>`.

Input:

پاکستان میں میں بارش ہوئی  
2024  

Output:

پاکستان | میں | میں | بارش | ہوئی  
`<NUM>`

In [3]:
# Your code here, Implement custom Urdu Tokenizer
import re
with open("cleaned.txt", "r", encoding="utf-8") as f:
    cleaned_text = f.read()
def urdu_tokenizer(text):
   # keeping only urdu words 
    # Remove punctuation except sentence markers
    text=re.sub(r"[،؛:“”()\[\]]", " ", text)
    # Split on whitespace
    tokens = text.split()
    return tokens
tokens = urdu_tokenizer(cleaned_text)

# testing if the tokenization worked
print("Total tokens:", len(tokens))
print("First 50 tokens:")
print(tokens[:50])

Total tokens: 358610
First 50 tokens:
['1', 'تصویر', 'کا', 'ذریعہ', 'تقریبا', 'سال', 'بعد', 'پاکستان', 'کے', 'صوبہ', 'پنجاب', 'کے', 'شہر', 'لاہور', 'میں', 'بسنت', 'کی', 'واپسی', 'پر', 'جشن', 'کا', 'ماحول', 'ہے۔', 'جہاں', 'ایک', 'طرف', 'صوبے', 'میں', 'تعطیل', 'کا', 'اعلان', 'کیا', 'گیا', 'اور', 'شہر', 'کے', 'مختلف', 'مقامات', 'کو', 'سجایا', 'گیا', 'تو', 'وہیں', 'بازاروں', 'میں', 'پتنگ', 'ڈور', 'اور', 'دیگر', 'اشیا']


## ***Custom Urdu Stemmer***
The stemmer must remove common Urdu suffixes.

Input:
لڑکیوں نے کتابیں پڑھیں  

Output:
لڑکی | کتاب | پڑھ


In [4]:
# Your code here, Implement custom Urdu Stemmer

#using some urdu words as a test
urdu_suffixes = ["وں", "یں", "ات", "یاں", "یاں", "گی", "گا","کے", "کی", "کا", "نے", "سے", "پر", "میں"]
def urdu_stemmer(tokens):
    # removing common suffixes 
    stemmed_tokens = []

    for word in tokens:
        original_word=word
        for suffix in urdu_suffixes:
            if word.endswith(suffix) and len(word)> len(suffix)+ 1:
                word =word[:-len(suffix)]
                break   # remove only one suffix

        stemmed_tokens.append(word)

    return stemmed_tokens

stemmed_tokens = urdu_stemmer(tokens)

# testing out the results 
print("Before stemming:", tokens[:30])
print("After stemming:", stemmed_tokens[:30])


Before stemming: ['1', 'تصویر', 'کا', 'ذریعہ', 'تقریبا', 'سال', 'بعد', 'پاکستان', 'کے', 'صوبہ', 'پنجاب', 'کے', 'شہر', 'لاہور', 'میں', 'بسنت', 'کی', 'واپسی', 'پر', 'جشن', 'کا', 'ماحول', 'ہے۔', 'جہاں', 'ایک', 'طرف', 'صوبے', 'میں', 'تعطیل', 'کا']
After stemming: ['1', 'تصویر', 'کا', 'ذریعہ', 'تقریبا', 'سال', 'بعد', 'پاکستان', 'کے', 'صوبہ', 'پنجاب', 'کے', 'شہر', 'لاہور', 'میں', 'بسنت', 'کی', 'واپسی', 'پر', 'جشن', 'کا', 'ماحول', 'ہے۔', 'جہاں', 'ایک', 'طرف', 'صوبے', 'میں', 'تعطیل', 'کا']


## ***Custom Urdu Lemmatizer***

The lemmatizer is restricted to:
- Plural normalization
- Gender normalization

Plural:
- لڑکیاں -> لڑکی  
- کتابوں -> کتاب  

Gender:
- اچھی -> اچھا  
- بڑی -> بڑا

In [5]:
# Your code here, Implement custom Urdu Lemmatizer
lemma_suffix_rules ={"یاں": "ی","یں": "","ات": "","وں": "","ے": "ہ"}   # making a dictionary for lemmatizing function

def urdu_lemmatizer(tokens):
   # normalize common urdu words
    lemmatized_tokens = []
    for word in tokens:
        original = word
        for suffix, replacement in lemma_suffix_rules.items():
            if word.endswith(suffix) and len(word)>len(suffix) + 1:
                word = word[:-len(suffix)] + replacement
                break

        lemmatized_tokens.append(word)
    return lemmatized_tokens
lemmatized_tokens=urdu_lemmatizer(stemmed_tokens)
# testing the function 
print("After stemming:", stemmed_tokens[:30])
print("After lemmatization:", lemmatized_tokens[:30])


After stemming: ['1', 'تصویر', 'کا', 'ذریعہ', 'تقریبا', 'سال', 'بعد', 'پاکستان', 'کے', 'صوبہ', 'پنجاب', 'کے', 'شہر', 'لاہور', 'میں', 'بسنت', 'کی', 'واپسی', 'پر', 'جشن', 'کا', 'ماحول', 'ہے۔', 'جہاں', 'ایک', 'طرف', 'صوبے', 'میں', 'تعطیل', 'کا']
After lemmatization: ['1', 'تصویر', 'کا', 'ذریعہ', 'تقریبا', 'سال', 'بعد', 'پاکستان', 'کے', 'صوبہ', 'پنجاب', 'کے', 'شہر', 'لاہور', 'میں', 'بسنت', 'کی', 'واپسی', 'پر', 'جشن', 'کا', 'ماحول', 'ہے۔', 'جہاں', 'ایک', 'طرف', 'صوبہ', 'میں', 'تعطیل', 'کا']


### ***Mandatory Deliverables for Part 1***
- JSON metadata file
- raw.txt
- cleaned.txt
- Tokenized dataset
- Stemmed dataset
- Lemmatized dataset


In [6]:
# Your code here, verify deliverables
import json
import re

#Load metadata
with open("Metadata.json", "r", encoding="utf-8") as f:
    metadata=json.load(f)

#Load raw text
with open("raw.txt","r",encoding="utf-8") as f:
    raw_text = f.read()

#Load cleaned text
with open("cleaned.txt","r",encoding="utf-8") as f:
    cleaned_text = f.read()

print("Number of articles in Metadata.json:", len(metadata))
print("Raw text size:", len(raw_text))
print("Cleaned text size:", len(cleaned_text))

article_numbers = re.findall(r"\[(\d+)\]", cleaned_text)
articles = re.split(r"\[\d+\]", cleaned_text)[1:]

print("Number of cleaned articles:", len(articles))

tokenized_dataset=[]

for article in articles:
    tokens = urdu_tokenizer(article)
    tokenized_dataset.append(tokens)

print("Tokenized dataset created.")
print("Sample tokens from article 1:")
print(tokenized_dataset[0][:30])

stemmed_dataset = []

for tokens in tokenized_dataset:
    stemmed = urdu_stemmer(tokens)
    stemmed_dataset.append(stemmed)

print("Stemmed dataset created.")
print("Sample stemmed tokens from article 1:")
print(stemmed_dataset[0][:30])

lemmatized_dataset = []

for tokens in stemmed_dataset:
    lemmas = urdu_lemmatizer(tokens)
    lemmatized_dataset.append(lemmas)

print("Lemmatized dataset created.")
print("Sample lemmatized tokens from article 1:")
print(lemmatized_dataset[0][:30])

Number of articles in Metadata.json: 285
Raw text size: 1721387
Cleaned text size: 1642241
Number of cleaned articles: 285
Tokenized dataset created.
Sample tokens from article 1:
['تصویر', 'کا', 'ذریعہ', 'تقریبا', 'سال', 'بعد', 'پاکستان', 'کے', 'صوبہ', 'پنجاب', 'کے', 'شہر', 'لاہور', 'میں', 'بسنت', 'کی', 'واپسی', 'پر', 'جشن', 'کا', 'ماحول', 'ہے۔', 'جہاں', 'ایک', 'طرف', 'صوبے', 'میں', 'تعطیل', 'کا', 'اعلان']
Stemmed dataset created.
Sample stemmed tokens from article 1:
['تصویر', 'کا', 'ذریعہ', 'تقریبا', 'سال', 'بعد', 'پاکستان', 'کے', 'صوبہ', 'پنجاب', 'کے', 'شہر', 'لاہور', 'میں', 'بسنت', 'کی', 'واپسی', 'پر', 'جشن', 'کا', 'ماحول', 'ہے۔', 'جہاں', 'ایک', 'طرف', 'صوبے', 'میں', 'تعطیل', 'کا', 'اعلان']
Lemmatized dataset created.
Sample lemmatized tokens from article 1:
['تصویر', 'کا', 'ذریعہ', 'تقریبا', 'سال', 'بعد', 'پاکستان', 'کے', 'صوبہ', 'پنجاب', 'کے', 'شہر', 'لاہور', 'میں', 'بسنت', 'کی', 'واپسی', 'پر', 'جشن', 'کا', 'ماحول', 'ہے۔', 'جہاں', 'ایک', 'طرف', 'صوبہ', 'میں', 'تعطیل', 'کا', 'اعل

## **Part 2: BBC Style Urdu News Article Generation**

Students must implement the following statistical language models:
- Unigram Language Model (for fallback and evaluation purposes)
- Bigram Language Model (for fallback and evaluation purposes)
- Trigram Language Model with fallback (Trigram → Bigram → Unigram)

The article generation system must use a fallback mechanism such that when a higher-order n-gram is unavailable, the model automatically backs off to a lower-order model.

Only Laplace (Add-One) smoothing is allowed for probability estimation.  
No other smoothing methods may be used.

In [7]:
# preparing data 
import re

# Represents start of sentence
#Needed for bigram & trigram models
START="<s>"

# Load cleaned text
with open("cleaned.txt", "r", encoding="utf-8") as f:
    cleaned_text = f.read()

# Remove article numbers for better model training 
text = re.sub(r"\[\d+\]", "", cleaned_text)

sentences=[]   # storing each sentence as a list of words ready for unigram, bigram, and trigram models.

# Split into sentences using Urdu punctuation for perplexity calculation
raw_sentences= re.split(r"[۔?!]", text)

# Each sentence is processed independently and removes whitespace in case any left 
for sent in raw_sentences:
    sent = sent.strip()
    if not sent:
        continue
    # Tokenizing the words 
    tokens = urdu_tokenizer(sent)
    # Stemming the words for better generalization 
    tokens =urdu_stemmer(tokens)
    # Lemmatize the sentences 
    tokens =urdu_lemmatizer(tokens)
   # Remove numeric tokens so numerical tokens dont get probability values 
    tokens=[t for t in tokens if t and not t.isdigit()]

    if len(tokens)>1:
        sentences.append([START, START] + tokens)

print("Total sentences for language modeling:", len(sentences))
print("Sample sentence:", sentences[0])


Total sentences for language modeling: 13431
Sample sentence: ['<s>', '<s>', 'تصویر', 'کا', 'ذریعہ', 'تقریبا', 'سال', 'بعد', 'پاکستان', 'کے', 'صوبہ', 'پنجاب', 'کے', 'شہر', 'لاہور', 'میں', 'بسنت', 'کی', 'واپسی', 'پر', 'جشن', 'کا', 'ماحول', 'ہے']


In [10]:
# Your code here, Train Unigram, Bigram and Trigram language models
from collections import defaultdict
import random
import math

random.seed(42)
# Use a subset for evaluation 
eval_sentences = random.sample(sentences, 200)

print("Number of evaluation sentences:", len(eval_sentences))
# vocab is being used to train the model, it will store all unique words and will use for smoothing 
vocab=set()

#dictionary like object to ensure the key value is 0 whever a new value is formed
unigram_counts = defaultdict(int)
bigram_counts = defaultdict(int)
trigram_counts = defaultdict(int)
# denominators for bi and trigram formulas 
bigram_context_counts = defaultdict(int)
trigram_context_counts = defaultdict(int)

# Build counts using sentences already having uni, bi and trigrams
for sentence in sentences:
    # building vocabulary by counting words appearance 
    for word in sentence:
        unigram_counts[word]+=1
        vocab.add(word)
    # creating Bigrams
    for i in range(len(sentence)-1):
        bigram=(sentence[i], sentence[i+1])
        bigram_counts[bigram]+= 1
        bigram_context_counts[sentence[i]]+= 1
    # creating Trigrams
    for i in range(len(sentence)-2):
        trigram =(sentence[i],sentence[i+1],sentence[i+2])
        trigram_counts[trigram]+=1
        trigram_context_counts[(sentence[i], sentence[i+1])]+= 1

V = len(vocab)  # size of vocabulary
total_unigrams = sum(unigram_counts.values()) #total word count

# setting up k smoothing value 
k = 0.001
bigram_probs_k = {} 
# applying bigram formula 
for (w1, w2),count in bigram_counts.items():
    prob=(count + k)/(bigram_context_counts[w1] + k * V)
    bigram_probs_k[(w1, w2)]= prob

trigram_probs_k = {}
# trigram formula 
for (w1, w2, w3), count in trigram_counts.items():
    context=(w1, w2)
    prob=(count + k)/(trigram_context_counts[context]+k*V)
    trigram_probs_k[(w1, w2, w3)]=prob

# for evaluation mfinding out perplexity by using log probabilities
def bigram_perplexity_k(sentences):
    log_prob_sum = 0
    count = 0
    for sentence in sentences:
        for i in range(1, len(sentence)):
            w1, w2=sentence[i-1], sentence[i]
            if w1 == "<s>" or w2 == "<s>":
                continue
            prob = bigram_probs_k.get((w1, w2),k/(bigram_context_counts.get(w1, 0)+k*V))
            log_prob_sum += math.log(prob)
            count += 1
    return math.exp(-log_prob_sum / count)
# perplexity of trigrams calculation 
def trigram_perplexity_k(sentences):
    log_prob_sum = 0
    count = 0
    for sentence in sentences:
        for i in range(2, len(sentence)):
            w1, w2, w3 =sentence[i-2], sentence[i-1], sentence[i]

            if w1 == "<s>" or w2 == "<s>" or w3 == "<s>":
                continue

            prob = trigram_probs_k.get((w1, w2, w3),k/(trigram_context_counts.get((w1, w2), 0) + k * V))
            log_prob_sum += math.log(prob)
            count += 1
    return math.exp(-log_prob_sum / count)

bi_pp_k  = bigram_perplexity_k(eval_sentences)
tri_pp_k = trigram_perplexity_k(eval_sentences)
print("Vocabulary size:", V)
print("Total unigrams:", total_unigrams)
print("Sample unigram count:", list(unigram_counts.items())[:5])

Number of evaluation sentences: 200
Vocabulary size: 12168
Total unigrams: 385184
Sample unigram count: [('<s>', 26862), ('تصویر', 1080), ('کا', 6406), ('ذریعہ', 1135), ('تقریبا', 145)]


In [11]:
unigram_probs = {}
for word in vocab:
    unigram_probs[word] = (unigram_counts[word] + 1) / (total_unigrams + V)

In [12]:
bigram_probs = {}
for (w1, w2), count in bigram_counts.items():
    bigram_probs[(w1, w2)] = (count + 1) / (bigram_context_counts[w1] + V)

In [13]:
trigram_probs = {}
for (w1, w2, w3), count in trigram_counts.items():
    trigram_probs[(w1, w2, w3)] = (count + 1) / (trigram_context_counts[(w1, w2)] + V)

In [14]:
# converting all n gram counts into probabilities using Laplace and K smoothing 
sample_word = list(vocab)[0]
print("Unigram probability sample:", unigram_probs[sample_word])

sample_bigram = list(bigram_probs.items())[:1]
print("Bigram probability sample:", sample_bigram)

sample_trigram = list(trigram_probs.items())[:1]
print("Trigram probability sample:", sample_trigram)

Unigram probability sample: 2.01332823290181e-05
Bigram probability sample: [(('<s>', '<s>'), 0.34414552908019475)]
Trigram probability sample: [(('<s>', '<s>', 'تصویر'), 0.0375014649009727)]


## ***Seed Prompt and Generation Constraints***

- Input must contain 5–8 Urdu words
- Single-word prompts are not allowed

Valid:
پاکستان میں مہنگائی کی شرح میں  

Invalid:
پاکستان  

### ***Article Length Rules***
- Minimum 150 words
- Target up to 200 words
- Minimum 5 sentences
- Hard stop at 300 words
- Forced termination if EOS is not produced

### ***Article Output Requirements***
Students must generate:
- Three complete Urdu news articles
- Five Urdu news headlines

Generated content must not copy full sentences from the training data.

In [15]:
# Your code here, Implement article generation logic
import random

#Randomly selects a word based on probability and prevents repetition
def sample_from_dict(prob_dict):
    words = list(prob_dict.keys())
    probs = list(prob_dict.values())
    return random.choices(words, weights=probs, k=1)[0]
# Implements fallback logic
def get_next_word(prev2, prev1):
    candidates = {}
    #Try TRIGRAM
    for (w1, w2, w3), prob in trigram_probs.items():
        if w1 == prev2 and w2 == prev1:
            candidates[w3] = prob
    if candidates:
        return sample_from_dict(candidates)
    #Fallback to BIGRAM
    for (w1, w2), prob in bigram_probs.items():
        if w1 == prev1:
            candidates[w2] = prob
    if candidates:
        return sample_from_dict(candidates)
    #Fallback to UNIGRAM
    return sample_from_dict(unigram_probs)
#generates one urdu sentence and forces EOF restrain
def generate_one_sentence(start_w1, start_w2, model="trigram", max_words=25):
    sentence =[start_w1, start_w2]
    for _ in range(max_words):
        next_word = get_next_word_by_model(sentence[-2], sentence[-1], model)
        sentence.append(next_word)
        if next_word == "۔":
            break
    if sentence[-1] != "۔":
        sentence.append("۔")
    return " ".join(sentence)
    
def generate_multiple_sentences(start_w1, start_w2, num_sentences=5):
    article = []
    current_w1, current_w2 = start_w1, start_w2

    for _ in range(num_sentences):
        sentence = generate_one_sentence(current_w1, current_w2)
        article.append(sentence)

        # Reset seed using last two words of the sentence
        tokens = sentence.split()
        if len(tokens) >= 2:
            current_w1, current_w2 = tokens[-2], tokens[-1]
        else:
            current_w1, current_w2 = start_w1, start_w2

    return "\n".join(article)

    # generates full articles using limits 
def generate_full_article(prompt, model="trigram",
                          min_words=150, max_words=300, min_sentences=5):

    tokens = urdu_tokenizer(prompt)

    if len(tokens) < 5 or len(tokens) > 8:
        raise ValueError("Prompt must contain 5–8 Urdu words")

    current_w1, current_w2 = tokens[-2], tokens[-1]

    article_sentences = []
    total_words = 0

    while True:
        sentence = generate_one_sentence(
            current_w1, current_w2, model=model
        )

        article_sentences.append(sentence)
        words = sentence.split()
        total_words += len(words)

        if len(words) >= 2:
            current_w1, current_w2 = words[-2], words[-1]

        if (
            len(article_sentences) >= min_sentences
            and total_words >= min_words
        ):
            break

        if total_words >= max_words:
            break

    return "\n".join(article_sentences), total_words

#Generates short Urdu headlines
def generate_headline(start_w1, start_w2, max_words=8):
    headline = [start_w1, start_w2]
    for _ in range(max_words - 2):
        next_word = get_next_word(headline[-2], headline[-1])
        if next_word == "۔":
            break
        headline.append(next_word)

    return " ".join(headline)
#Allows model selection
def get_next_word_by_model(prev2, prev1, model="trigram"):
    candidates = {}
    if model == "trigram":
        # Trigram → Bigram → Unigram
        return get_next_word(prev2, prev1)
    elif model == "bigram":
        # Bigram → Unigram
        for (w1, w2), prob in bigram_probs.items():
            if w1 == prev1:
                candidates[w2] = prob
        if candidates:
            return sample_from_dict(candidates)
        return sample_from_dict(unigram_probs)
    else:
        raise ValueError("Model must be 'bigram' or 'trigram'")
prompts = ["ملکی منظرنامے میں بدلتے ہوئے رجحانات","عوامی ردعمل اور پالیسی فیصلوں کا اثر","حالیہ واقعات کے پس منظر میں تجزیہ"]
articles = []
for i, prompt in enumerate(prompts, 1):
    article, wc = generate_full_article(prompt)
    articles.append(article)

    print(f"\n================ ARTICLE {i} ================\n")
    print("Prompt:", prompt)
    print(article)
    print("\nWord count:", wc)

headline_seeds = [("حالیہ", "پیشرفت"),("عوامی", "سطح"),("سیاسی", "تناظر"),("قومی", "بحث"),("سماجی", "حالات")]

print("\n=========== GENERATED HEADLINES ===========\n")
for i, (w1, w2) in enumerate(headline_seeds, 1):
    print(f"{i}.", generate_headline(w1, w2))


================ ARTICLE 1 ================

Prompt: ملکی منظرنامے میں بدلتے ہوئے رجحانات
ہوئے رجحانات اللہ بیگ نے لکھا ایک غریب اور محنت کش خاندان کے بارہ میں ابھی بہت جان ہے وہ بہرحال اپنی جگہ ہوتا رہا اور ٹیم ۔
ٹیم ۔ ہونا چاہی کی کہ زمین پر گر گئ اور سخت کام کرنا چاہیہ اور کیمیکل یا دھاتی ڈور استعمال کر رہہ تھہ کہ حیدرآباد ریاست ۔
ریاست ۔ کے موقع پر پاکستانی کپتان سلمان علی آغا تین گیند مسلسل آف ستمپ کے باہر گھیر لیا ہسپتال کی پرچ اور ان کے منھ پر ۔
پر ۔ کی جائہ اور بی سی سپورٹس نے سرہ سے جھٹلا اور بعد میں مجھہ یہ اعتماد کی بحالی شروع ہوئی تھی تو شاہد آفریدی کا ۔
کا ۔ جبکہ صرف پانچ فٹ آٹھ انچ ہے جو مبینہ طور پر زیادہ غم و غصہ کا اظہار کیا کہ فرنچائز کو آئندہ پانچ سیزنز میں ۔
میں ۔ میں شامل ہو جاتا ہے اس بک میں آپ یہ کی کیا؟ ہم نے بہت شوق سے مجھہ جوناگڑھ کی نواب بیگم کے طور پر ۔

Word count: 168

================ ARTICLE 2 ================

Prompt: عوامی ردعمل اور پالیسی فیصلوں کا اثر
کا اثر کھیل کے طور پر ایک ویڈیو میں اس بات کا اعلان کیا تھا کہ ٹی انٹرنیشنل میں آٹھ وکٹ کے نقصان پر

### ***UI Requirements (Bonus)***
The system must allow:
- Model selection (Bigram / Trigram)
- Seed input
- Article generation
- Proper Right-to-Left Urdu display


In [26]:
# Your code here, Implement simple interactive UI
print("=== Urdu News Article Generator ===")
print("Select Language Model:")
print("1. Bigram")
print("2. Trigram")

choice = input("Enter choice (1 or 2): ").strip()

if choice == "1":
    model_choice = "bigram"
elif choice == "2":
    model_choice = "trigram"
else:
    print("Invalid choice. Defaulting to trigram.")
    model_choice = "trigram"

seed = input("Enter 5–8 word Urdu seed: ").strip()

try:
    article, wc = generate_full_article(seed, model=model_choice)
    print("\n=== Generated Article ===\n")
    print(article)
    print("\nWord count:", wc)
    print("Model used:", model_choice)
except Exception as e:
    print("Error:", e)

=== Urdu News Article Generator ===
Select Language Model:
1. Bigram
2. Trigram


Enter choice (1 or 2):  2
Enter 5–8 word Urdu seed:  میں تجزیہ کار سلیم خالق نے لکھا کہ



=== Generated Article ===

لکھا کہ دشمنتھا چمیرا نے اپنا سٹروک پلہ جاری رکھا اور پاکستان کے کون میں میدان جنگ میں بہادری سے مقابلہ طے کر کے کم از کم ۔
کم ۔ کے لیہ رکھ لے اور میری ویڈیوز میں موجود ہوں وہاں سیاست کا قبضہ ہو بلکہ پاکستان کے شہر بلاوائیو میں کھیلہ گئہ ہیں جو ۔
جو ۔ بغیر کسی خاص لباس یا جم کی ممبرشپ کے بھی سافٹ ویئر درست کام نہ دیں گے تو مقابلہ کی صورت میں رائٹ آرم آف ۔
آف ۔ لڑک جو مختلف ڈیجیٹل پلیٹ فارمز تک بھی پھیل رہا ہے اور میجر سنگھ ایک ڈائریکٹر کے طور پر میں شائع تحریر کا جائزہ لیا ۔
لیا ۔ سی بی یہ دلیل دے سکتا ہوں جنھ نے خواتین پر مبنی ہیں اور ٹی میچز کھیل چکی ہے تو کئی شائقین نے مایوسی ظاہر ۔
ظاہر ۔ کرہ گی بلکہ معیاری کرکٹ نے دنیا کو نظر انداز کرتہ ہوئہ سپورٹس جرنلسٹ وکرانت گپتا نے ایکس پر انڈین کرکٹر عرفان پٹھان کی بات ۔

Word count: 168
Model used: trigram


## ***Evaluation and Analysis***

Students must perform the following evaluations:
- Display generated articles for comparison
- Quantitative evaluation using perplexity scores

In [16]:
# Your code here, Implement evaluation functions and graph plots
import math
import random

START = "<s>"

# Use a small subset of sentences for evaluation
random.seed(42)
eval_sentences = random.sample(sentences, 200)

print("Number of evaluation sentences:", len(eval_sentences))


def unigram_perplexity(sentences):
    log_prob_sum = 0
    word_count = 0
    for sentence in sentences:
        for word in sentence:
            if word == START:
                continue  # ignore start tokens
            prob = unigram_probs.get(word, 1 / (total_unigrams + V))
            log_prob_sum += math.log(prob)
            word_count += 1

    return math.exp(-log_prob_sum / word_count)


def bigram_perplexity(sentences):
    log_prob_sum = 0
    count = 0

    for sentence in sentences:
        if len(sentence) < 2:
            continue

        for i in range(1, len(sentence)):
            w1, w2 = sentence[i-1], sentence[i]

            if w1 == START or w2 == START:
                continue  # ignore start tokens

            if (w1, w2) in bigram_probs:
                prob = bigram_probs[(w1, w2)]
            else:
                prob = unigram_probs.get(w2, 1 / (total_unigrams + V))

            log_prob_sum += math.log(prob)
            count += 1

    return math.exp(-log_prob_sum / count)


def trigram_perplexity(sentences):
    log_prob_sum = 0
    count = 0

    for sentence in sentences:
        if len(sentence) < 3:
            continue

        for i in range(2, len(sentence)):
            w1, w2, w3 = sentence[i-2], sentence[i-1], sentence[i]

            if w1 == START or w2 == START or w3 == START:
                continue

            if (w1, w2, w3) in trigram_probs:
                prob = trigram_probs[(w1, w2, w3)]
            elif (w2, w3) in bigram_probs:
                prob = bigram_probs[(w2, w3)]
            else:
                prob = unigram_probs.get(w3, 1 / (total_unigrams + V))

            log_prob_sum += math.log(prob)
            count += 1

    return math.exp(-log_prob_sum / count)


uni_pp = unigram_perplexity(eval_sentences)
bi_pp  = bigram_perplexity(eval_sentences)
tri_pp = trigram_perplexity(eval_sentences)

print("\n")
print("Unigram:", round(uni_pp, 2))
print("Bigram :", round(bi_pp, 2))
print("Trigram:", round(tri_pp, 2))

print("\nPerplexity Results===")
print("Bigram (k):", round(bi_pp_k, 2))
print("Trigram (k):", round(tri_pp_k, 2))


Number of evaluation sentences: 200


Unigram: 762.87
Bigram : 859.56
Trigram: 2660.8

Perplexity Results===
Bigram (k): 42.51
Trigram (k): 13.09
